<a href="https://colab.research.google.com/github/shims79757-lang/Elevance-Skills-Projects/blob/main/App_Category_Performance_Analysis_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datetime import datetime
from zoneinfo import ZoneInfo
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from scipy.cluster.hierarchy import leaves_list, linkage
from scipy.stats import zscore

# 1. Load Dataset
url = "https://raw.githubusercontent.com/shims79757-lang/Elevance-Skills-Projects/main/googleplaystore.csv"
apps = pd.read_csv(url)
data = apps.copy()

# 2. Data Cleaning & Type Conversion
data["Rating"] = pd.to_numeric(data["Rating"], errors="coerce")
data["Reviews"] = pd.to_numeric(data["Reviews"], errors="coerce")

# Clean and convert Installs
data["Installs"] = (
    data["Installs"]
    .astype(str)
    .str.replace(",", "", regex=False)
    .str.replace("+", "", regex=False)
)
data["Installs"] = pd.to_numeric(data["Installs"], errors="coerce")


# Convert Size to MB
def convert_size(size):
  size = str(size).strip()
  if size.endswith("M"):
    return float(size[:-1])
  elif size.endswith("k"):
    return float(size[:-1]) / 1024
  return np.nan


data["Size_MB"] = data["Size"].apply(convert_size)

# Parse Last Updated date
data["Last Updated"] = pd.to_datetime(data["Last Updated"], errors="coerce")

# Deduplicate
data = data.drop_duplicates(subset=["App", "Category"]).reset_index(drop=True)

# 3. Filtering Criteria
# - Rating >= 4.0
# - Size > 10 MB
# - Installs >= 10,000
# - Reviews > 1,000
# - Last-updated month of January
# - Exclude app names containing numbers
filtered_data = data[
    (data["Rating"] >= 4.0)
    & (data["Size_MB"] > 10)
    & (data["Installs"] >= 10000)
    & (data["Reviews"] > 1000)
    & (data["Last Updated"].dt.month == 1)
    & (~data["App"].str.contains(r"\d", regex=True, na=False))
].copy()

# 4. Select Top 10 Eligible Categories (by Total Installs)
top_10_cats = (
    filtered_data.groupby("Category")["Installs"]
    .sum()
    .nlargest(10)
    .index.tolist()
)
cat_subset = filtered_data[filtered_data["Category"].isin(top_10_cats)].copy()


# 5. Compute the 6 Required Metrics per Category
def compute_metrics(group):
  # Weighted Rating = sum(Rating * Reviews) / sum(Reviews)
  weighted_rating = (
      (group["Rating"] * group["Reviews"]).sum() / group["Reviews"].sum()
      if group["Reviews"].sum() > 0
      else group["Rating"].mean()
  )
  total_reviews = group["Reviews"].sum()
  total_installs = group["Installs"].sum()
  avg_size = group["Size_MB"].mean()
  # Engagement Rate = (Total Reviews / Total Installs) * 100
  engagement_rate = (
      (total_reviews / total_installs * 100) if total_installs > 0 else 0
  )
  # Update Frequency = Proportion of apps updated in recent period (%)
  latest_year = group["Last Updated"].dt.year.max()
  update_freq = (group["Last Updated"].dt.year == latest_year).mean() * 100

  return pd.Series({
      "Weighted Rating": weighted_rating,
      "Total Reviews": total_reviews,
      "Total Installs": total_installs,
      "Average Size (MB)": avg_size,
      "Engagement Rate (%)": engagement_rate,
      "Update Frequency (%)": update_freq,
  })


raw_df = cat_subset.groupby("Category").apply(compute_metrics)

# 6. Z-Score Normalization
norm_df = raw_df.apply(zscore)

# 7. Dynamic Ranking using Weighted Composite Score
weights = pd.Series({
    "Weighted Rating": 0.20,
    "Total Reviews": 0.15,
    "Total Installs": 0.25,
    "Average Size (MB)": 0.10,
    "Engagement Rate (%)": 0.15,
    "Update Frequency (%)": 0.15,
})
composite_scores = norm_df.dot(weights)
raw_df["Composite_Score"] = composite_scores
norm_df["Composite_Score"] = composite_scores

# 8. Hierarchical Clustering (Ward's Linkage on Normalized Metrics)
metrics_cols = [c for c in norm_df.columns if c != "Composite_Score"]
linkage_matrix = linkage(norm_df[metrics_cols], method="ward", metric="euclidean")
cluster_order = leaves_list(linkage_matrix)

# Reorder categories according to hierarchical clustering
ordered_categories = norm_df.index[cluster_order]
raw_df_ordered = raw_df.loc[ordered_categories]
norm_df_ordered = norm_df.loc[ordered_categories]
composite_scores_ordered = composite_scores.loc[ordered_categories]

# Identify 3 Highest and 3 Lowest Composite Scores
top_3_cats = composite_scores.nlargest(3).index.tolist()
bottom_3_cats = composite_scores.nsmallest(3).index.tolist()

# Label annotations on y-axis
y_labels = []
for cat in ordered_categories:
  score = composite_scores_ordered[cat]
  if cat in top_3_cats:
    rank = top_3_cats.index(cat) + 1
    y_labels.append(f"{cat} ★ [Top {rank}: {score:+.2f}]")
  elif cat in bottom_3_cats:
    rank = 3 - bottom_3_cats.index(cat)
    y_labels.append(f"{cat} ▼ [Low {rank}: {score:+.2f}]")
  else:
    y_labels.append(f"{cat} ({score:+.2f})")

# Prepare Hover & Cell Text
norm_vals = norm_df_ordered[metrics_cols].values
raw_vals = raw_df_ordered[metrics_cols].values

text_norm = np.round(norm_vals, 2).astype(str)
text_raw = np.array([[f"{v:,.1f}" for v in row] for row in raw_vals])

# 9. Build Interactive Clustered Heatmap
fig = go.Figure()

fig.add_trace(
    go.Heatmap(
        z=norm_vals,
        x=metrics_cols,
        y=y_labels,
        text=text_norm,
        texttemplate="%{text}",
        colorscale="Viridis",
        colorbar=dict(title="Z-Score"),
        hoverongaps=False,
    )
)

# Switch Selector between Normalized and Raw Values
fig.update_layout(
    title=dict(
        text=(
            "<b>Clustered Heatmap of Top 10 App Categories</b><br><sup>Arranged"
            " via Hierarchical Clustering | Annotations: ★ Top 3, ▼ Bottom 3"
            " Composite Scores</sup>"
        ),
        x=0.5,
    ),
    template="plotly_white",
    height=650,
    width=950,
    xaxis=dict(title="Performance Metrics", tickangle=-20),
    yaxis=dict(title="Categories (Hierarchically Clustered)"),
    updatemenus=[
        dict(
            type="buttons",
            direction="left",
            x=0.0,
            y=1.16,
            showactive=True,
            buttons=[
                dict(
                    label="Normalized (Z-Score)",
                    method="update",
                    args=[
                        {
                            "z": [norm_vals],
                            "text": [text_norm],
                            "colorscale": "Viridis",
                            "colorbar": [{"title": "Z-Score"}],
                        },
                        {
                            "title": (
                                "<b>Clustered Heatmap of Top 10 App"
                                " Categories (Z-Score Normalized)</b>"
                            )
                        },
                    ],
                ),
                dict(
                    label="Raw Values",
                    method="update",
                    args=[
                        {
                            "z": [raw_vals],
                            "text": [text_raw],
                            "colorscale": "Blues",
                            "colorbar": [{"title": "Raw Value"}],
                        },
                        {
                            "title": (
                                "<b>Clustered Heatmap of Top 10 App"
                                " Categories (Raw Metrics)</b>"
                            )
                        },
                    ],
                ),
            ],
        )
    ],
)

# 10. IST Time-Restriction Condition (3:00 PM – 5:00 PM IST)
current_time = datetime.now(ZoneInfo("Asia/Kolkata"))
print("Current IST:", current_time.strftime("%d %B %Y, %I:%M %p"))

if 15 <= current_time.hour < 17:
  fig.show()
else:
  print(
      "Visualization unavailable.\n"
      "This graph can only be viewed between 3:00 PM and 5:00 PM IST."
  )